<a href="https://colab.research.google.com/github/GautamAjesh/A-Comparative-Analysis-of-Quantization-Methods-Across-NLP-Tasks-in-Small-Language-Models/blob/main/quantization_nlp_experiments_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch, os, time, re, string
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForQuestionAnswering
from datasets import load_dataset

# ===== DistilBERT models =====
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

tokenizer_nli = AutoTokenizer.from_pretrained("textattack/distilbert-base-uncased-RTE")
model_nli = AutoModelForSequenceClassification.from_pretrained("textattack/distilbert-base-uncased-RTE")

tokenizer_qa = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
model_qa = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")

quantized_model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_nli = torch.quantization.quantize_dynamic(model_nli, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_qa = torch.quantization.quantize_dynamic(model_qa, {torch.nn.Linear}, dtype=torch.qint8)

# ===== ALBERT models =====
tokenizer_alb_sent = AutoTokenizer.from_pretrained("textattack/albert-base-v2-SST-2")
model_alb_sent = AutoModelForSequenceClassification.from_pretrained("textattack/albert-base-v2-SST-2")

tokenizer_alb_qa = AutoTokenizer.from_pretrained("Firat/albert-base-v2-finetuned-squad")
model_alb_qa = AutoModelForQuestionAnswering.from_pretrained("Firat/albert-base-v2-finetuned-squad")

tokenizer_alb_nli2 = AutoTokenizer.from_pretrained("Alireza1044/albert-base-v2-rte")
model_alb_nli2 = AutoModelForSequenceClassification.from_pretrained("Alireza1044/albert-base-v2-rte")
model_alb_nli2.eval()

quantized_model_alb_sent = torch.quantization.quantize_dynamic(model_alb_sent, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_alb_qa = torch.quantization.quantize_dynamic(model_alb_qa, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_alb_nli2 = torch.quantization.quantize_dynamic(model_alb_nli2, {torch.nn.Linear}, dtype=torch.qint8)

# ===== Datasets (large versions) =====
dataset_large = load_dataset("stanfordnlp/sst2", split="validation[:500]")
dataset_rte_large = load_dataset("nyu-mll/glue", "rte", split="validation")
dataset_qa_large = load_dataset("rajpurkar/squad", split="validation[:500]")

# ===== Functions =====
def get_model_size(model, label="model"):
    torch.save(model.state_dict(), "temp.pt")
    size_mb = os.path.getsize("temp.pt") / (1024 * 1024)
    os.remove("temp.pt")
    print(f"{label} size: {size_mb:.2f} MB")
    return size_mb

def evaluate_model(model, dataset, tokenizer):
    correct = 0
    for example in dataset:
        inputs = tokenizer(example['sentence'], return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
        if pred == example['label']:
            correct += 1
    return correct / len(dataset)

def evaluate_nli_model(model, dataset, tokenizer):
    correct = 0
    for example in dataset:
        inputs = tokenizer(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
        if pred == example['label']:
            correct += 1
    return correct / len(dataset)

def get_answer(model, tokenizer, question, context):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384)
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx+1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)

def normalize_text(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)

def evaluate_qa_model(model, tokenizer, dataset):
    f1_scores = []
    for example in dataset:
        predicted = get_answer(model, tokenizer, example['question'], example['context'])
        best_f1 = max(compute_f1(predicted, ans) for ans in example['answers']['text'])
        f1_scores.append(best_f1)
    return sum(f1_scores) / len(f1_scores)

def measure_latency(predict_fn, n_runs=50):
    predict_fn()
    start = time.time()
    for _ in range(n_runs):
        predict_fn()
    end = time.time()
    return ((end - start) / n_runs) * 1000

print("Full environment restored!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

/tmp/ipykernel_2255/2315906262.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
/tmp/ipykernel_2255/2315906262.py:17: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.qua

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

/tmp/ipykernel_2255/2315906262.py:31: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model_alb_sent = torch.quantization.quantize_dynamic(model_alb_sent, {torch.nn.Linear}, dtype=torch.qint8)
/tmp/ipykernel_2255/2315906262.py:32: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantiza

Full environment restored!


In [6]:
for i in range(10):
    example = dataset_rte_large[i]
    inputs = tokenizer_alb_nli2(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model_alb_nli2(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    print(f"idx={i}, predicted={pred}, true={example['label']}, logits={outputs.logits.tolist()}")

idx=0, predicted=1, true=1, logits=[[-1.5901401042938232, 1.5687311887741089]]
idx=1, predicted=1, true=0, logits=[[-1.7683064937591553, 1.4724273681640625]]
idx=2, predicted=1, true=1, logits=[[-1.8220751285552979, 1.6654937267303467]]
idx=3, predicted=1, true=1, logits=[[-1.6598680019378662, 1.771700382232666]]
idx=4, predicted=1, true=0, logits=[[-1.912562608718872, 1.44669771194458]]
idx=5, predicted=1, true=0, logits=[[-2.1189398765563965, 1.3811923265457153]]
idx=6, predicted=1, true=0, logits=[[-1.9533480405807495, 1.5332858562469482]]
idx=7, predicted=1, true=0, logits=[[-1.8874319791793823, 1.5329670906066895]]
idx=8, predicted=1, true=0, logits=[[-1.5838608741760254, 1.5742249488830566]]
idx=9, predicted=1, true=0, logits=[[-1.6601283550262451, 1.6852734088897705]]


In [7]:
fp32_f1_alb_qa_full = evaluate_qa_model(model_alb_qa, tokenizer_alb_qa, dataset_qa_large)
int8_f1_alb_qa_full = evaluate_qa_model(quantized_model_alb_qa, tokenizer_alb_qa, dataset_qa_large)

print(f"ALBERT QA FP32 F1 (n=500): {fp32_f1_alb_qa_full*100:.2f}")
print(f"ALBERT QA INT8 F1 (n=500): {int8_f1_alb_qa_full*100:.2f}")
print(f"Drop: {(fp32_f1_alb_qa_full - int8_f1_alb_qa_full)*100:.2f} pts")

ALBERT QA FP32 F1 (n=500): 60.91
ALBERT QA INT8 F1 (n=500): 0.65
Drop: 60.26 pts


In [8]:
tokenizer_alb_nli3 = AutoTokenizer.from_pretrained("textattack/albert-base-v2-RTE")
model_alb_nli3 = AutoModelForSequenceClassification.from_pretrained("textattack/albert-base-v2-RTE")
model_alb_nli3.eval()

print("Label map:", model_alb_nli3.config.id2label)

for i in range(10):
    example = dataset_rte_large[i]
    inputs = tokenizer_alb_nli3(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model_alb_nli3(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    print(f"idx={i}, predicted={pred}, true={example['label']}, logits={outputs.logits.tolist()}")

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Label map: {0: 'LABEL_0', 1: 'LABEL_1'}
idx=0, predicted=1, true=1, logits=[[-1.8391889333724976, 1.3854418992996216]]
idx=1, predicted=1, true=0, logits=[[-1.9225131273269653, 1.5338948965072632]]
idx=2, predicted=1, true=1, logits=[[-1.8423466682434082, 1.3314318656921387]]
idx=3, predicted=1, true=1, logits=[[-1.9213578701019287, 1.3883153200149536]]
idx=4, predicted=1, true=0, logits=[[-1.73966646194458, 1.2196401357650757]]
idx=5, predicted=1, true=0, logits=[[-1.7422648668289185, 1.3691599369049072]]
idx=6, predicted=1, true=0, logits=[[-1.4844688177108765, 1.1839656829833984]]
idx=7, predicted=1, true=0, logits=[[-0.8854582905769348, 0.7007336616516113]]
idx=8, predicted=1, true=0, logits=[[-1.9173945188522339, 1.4569729566574097]]
idx=9, predicted=1, true=0, logits=[[-1.6562556028366089, 1.2951725721359253]]
